# 01 - Sentinel-2 Harmonized SR Collection Loading & Cloud Masking

**What:** Load the `COPERNICUS/S2_SR_HARMONIZED` surface-reflectance collection over an AOI and date range, then apply the full legacy filtering chain: tile-level cloud-percentage filter, AOI coverage filter, AOI-local valid-pixel filter via the Scene Classification Layer (SCL), SCL class masking, and a unique-day reduction. Along the way it accumulates the same `collection_info` metadata stages the plugin reports to the user.

**Why:** This is the data-ingestion backbone of the RAVI plugin. Every downstream vegetation-index / time-series step consumes `self.sentinel2` built here. Reproducing it in a notebook lets us validate the Earth Engine logic in isolation (server-side counts before/after every filter) without the QGIS UI.

**Legacy reference:** `ravi_dialog.py` (branch `legacy`) functions:
- `ee_load_collection` (~3604-3704) - collection build, date/bounds/cloud filtering, orchestration, `collection_info` stages.
- `uniqueday_collection` (~3704-3741) - one image per day, best AOI coverage.
- `AOI_coverage_filter` (~3742-3807) - `coverage_ratio` per image, gte threshold filter.
- `SCL_filter` / `mask_cloud_and_shadows` (~3808-3884) - per-AOI valid-pixel percentage via SCL.
- `SCL_mask` (~3886-3913) - hard SCL class mask applied to imagery.
- `apply_buffer` (~2483) - optional AOI buffering.

**UI controls mirrored:** `horizontalSlider_total_pixel_limit` (tile CLOUDY_PIXEL_PERCENTAGE), `horizontalSlider_aio_cover` (AOI coverage %), `horizontalSlider_local_pixel_limit` (AOI valid-pixel %), `mask_class0..mask_class11` checkboxes (SCL classes), `mask` checkbox (cloud+shadow mask), `horizontalSlider_buffer` (AOI buffer).

In [1]:
import os
import ee
import pandas as pd

# geemap is optional - only used for an interactive preview at the end.
try:
    import geemap
    HAS_GEEMAP = True
except ImportError:
    HAS_GEEMAP = False
    print("geemap not available - skipping interactive map preview.")

# Service-account auth via the GEE env var (same as dev/testing.ipynb).
credentials = ee.ServiceAccountCredentials(None, os.environ["GEE"])
ee.Initialize(credentials)
print("Earth Engine initialized.")

Earth Engine initialized.


In [2]:
import zipfile

import geopandas as gpd


def load_aoi_from_shapefile(shapefile_path):
    """Load an AOI from a (zipped) shapefile as an ee.FeatureCollection.

    Same loader as dev/testing.ipynb: reproject to EPSG:4326, dissolve to a
    single geometry, strip any Z coordinate, wrap in a FeatureCollection.
    """
    if shapefile_path.endswith(".zip"):
        with zipfile.ZipFile(shapefile_path, "r") as zip_ref:
            shapefile_within_zip = None
            for file in zip_ref.namelist():
                if file.lower().endswith(".shp"):
                    shapefile_within_zip = file
                    break
            if not shapefile_within_zip:
                raise FileNotFoundError(f"No .shp file found inside the zip archive: {shapefile_path}")
            gpd_aoi = gpd.read_file(f"zip://{shapefile_path}/{shapefile_within_zip}")
    else:
        gpd_aoi = gpd.read_file(shapefile_path)

    gpd_aoi = gpd_aoi.to_crs(epsg=4326)

    if gpd_aoi.empty:
        raise ValueError(f"The shapefile at {shapefile_path} does not contain any geometries.")

    if len(gpd_aoi) > 1:
        gpd_aoi = gpd_aoi.dissolve()

    geometry = gpd_aoi.geometry.iloc[0]
    geojson = geometry.__geo_interface__

    if geojson["type"] == "Polygon":
        geojson["coordinates"] = [
            list(map(lambda coord: coord[:2], ring)) for ring in geojson["coordinates"]
        ]
    elif geojson["type"] == "MultiPolygon":
        geojson["coordinates"] = [
            [list(map(lambda coord: coord[:2], ring)) for ring in polygon]
            for polygon in geojson["coordinates"]
        ]

    ee_geometry = ee.Geometry(geojson)
    feature = ee.Feature(ee_geometry)
    return ee.FeatureCollection([feature])


# Shared AOI loaded from the project shapefile (same area as dev/testing.ipynb).
# The legacy code treats the AOI as an ee.FeatureCollection (aoi.first().geometry(),
# aoi.map(...), feature.buffer(...)); aoi is the underlying geometry.
aoi_fc = load_aoi_from_shapefile("contorno_area_total.zip")
aoi = aoi_fc.geometry()
start, end = "2023-01-01", "2024-01-01"

# collection_info / collection_info_pt accumulate the human-readable stage
# metadata exactly like the plugin (English + Portuguese strings).
collection_info = []
collection_info_pt = []

## (Optional) AOI buffer - `apply_buffer`

Legacy `apply_buffer` reads `horizontalSlider_buffer` and, when non-zero, buffers every feature of the AOI FeatureCollection. We reproduce it with a configurable distance (default 0 = no buffer).

In [3]:
def apply_buffer(aoi_collection, buffer_distance=0):
    """Applies a buffer to the AOI geometry (legacy apply_buffer, ~line 2483)."""
    if buffer_distance != 0:
        print(f"Buffer distance: {buffer_distance} meters")
        return aoi_collection.map(lambda feature: feature.buffer(buffer_distance))
    print("No buffer applied")
    return aoi_collection


# horizontalSlider_buffer value. 0 keeps the AOI unchanged.
buffer_distance = 0
aoi_fc = apply_buffer(aoi_fc, buffer_distance)

No buffer applied


## Step 1 - Build the base collection + tile cloud filter (`ee_load_collection`)

Faithful to `ee_load_collection` (~3627-3666):
1. `ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")` filtered by date and AOI bounds, with a `date` property (`YYYY-MM-dd`) mapped onto each image.
2. Record the **size before any filtering** (`collection_info` stage).
3. Apply the tile-level cloud filter `ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", nuvem)` where `nuvem` comes from `horizontalSlider_total_pixel_limit`. Record size **after tile cloud filtering (Step 8)**.

We print `.size().getInfo()` before and after so the cloud filter effect is verifiable.

In [6]:
# UI inputs mirrored from ee_load_collection.
nuvem = 50                  # horizontalSlider_total_pixel_limit (CLOUDY_PIXEL_PERCENTAGE)
coverage_threshold = 0.90   # horizontalSlider_aio_cover.value() / 100
local_pixel_limit = 50      # horizontalSlider_local_pixel_limit.value()
mask_is_checked = True      # self.mask.isChecked() -> apply SCL_mask

inicio, final = start, end

sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate(inicio, final)
    .filterBounds(aoi)
    .map(lambda image: image.set("date", image.date().format("YYYY-MM-dd")))
)

original_count = sentinel2.size().getInfo()
print(f"Collection size before any filtering: {original_count}")
collection_info.append(f"Collection size before any filtering: {original_count}")
collection_info_pt.append(
    f"Tamanho da cole\u00e7\u00e3o antes de qualquer filtro: {original_count}"
)
assert original_count > 0, "No images for the AOI/date range - widen the criteria."

# Tile-level cloud percentage filter.
sentinel2 = sentinel2.filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", nuvem))
cloud_filtered_count = sentinel2.size().getInfo()
print(f"Collection size after cloud filtering: {cloud_filtered_count}")
collection_info.append(
    f"Collection size after tile cloud percentage filtering (Step 8): {cloud_filtered_count}"
)
collection_info_pt.append(
    f"Tamanho da cole\u00e7\u00e3o ap\u00f3s filtro de limite de nuvem na cena (Passo 8): {cloud_filtered_count}"
)

Collection size before any filtering: 288
Collection size after cloud filtering: 172


## Step 2 - AOI coverage filter (`AOI_coverage_filter`)

Faithful to `AOI_coverage_filter` (~3742-3807). For each image it computes `coverage_ratio = area(AOI ∩ image_footprint) / area(AOI)` using `ee.ErrorMargin(1)`, sets it as an image property, then keeps images with `ee.Filter.gte("coverage_ratio", coverage_threshold)`. A `coverage_threshold` of exactly 1 is nudged to `0.9999` to dodge float-comparison issues. Recorded as **AOI overlap filter (Step 6)**.

Only applied when `coverage_threshold > 0` (matches the `if coverage_threshold > 0:` guard in `ee_load_collection`).

In [7]:
def AOI_coverage_filter(sentinel2, aoi, coverage_threshold):
    """Filters by AOI coverage ratio (legacy AOI_coverage_filter, ~line 3742)."""
    print("Applying AOI coverage filter...")
    if coverage_threshold == 1:
        coverage_threshold = 0.9999  # Avoid floating-point comparison issues

    aoi_geometry = aoi.first().geometry()
    aoi_area = aoi_geometry.area()

    def calculate_coverage_ratio(image):
        intersection = aoi_geometry.intersection(image.geometry(), ee.ErrorMargin(1))
        intersection_area = intersection.area()
        coverage_ratio = intersection_area.divide(aoi_area)
        return image.set("coverage_ratio", coverage_ratio)

    sentinel2_with_ratio = sentinel2.map(calculate_coverage_ratio)
    coverage_filter = ee.Filter.gte("coverage_ratio", coverage_threshold)
    covering_colection = sentinel2_with_ratio.filter(coverage_filter)

    filtered_count = covering_colection.size().getInfo()
    print(
        f"Collection size after filtro >= {coverage_threshold * 100}% AOI coverage: {filtered_count}"
    )
    collection_info.append(
        f"Collection size after AOI overlap filter (Step 6): {filtered_count}"
    )
    collection_info_pt.append(
        f"Tamanho da cole\u00e7\u00e3o ap\u00f3s filtro de sobreposi\u00e7\u00e3o com a AOI (Passo 6): {filtered_count}"
    )
    return covering_colection


if coverage_threshold > 0:
    sentinel2 = AOI_coverage_filter(sentinel2, aoi_fc, coverage_threshold)
    assert sentinel2.size().getInfo() > 0, "AOI coverage filter emptied the collection."
else:
    print("coverage_threshold == 0 -> AOI coverage filter skipped (matches legacy guard).")

Applying AOI coverage filter...
Collection size after filtro >= 90.0% AOI coverage: 162


## Step 3 - AOI-local valid-pixel filter via SCL (`SCL_filter` / `mask_cloud_and_shadows`)

Faithful to `SCL_filter` (~3808-3884). The `scl_classes_behavior` dict maps the 12 SCL Scene-Classification classes to the `mask_class0..mask_class11` checkboxes:

| SCL | Class | UI checkbox |
|----|-------|-------------|
| 0 | No data | `mask_class0` |
| 1 | Saturated / defective | `mask_class1` |
| 2 | Dark features | `mask_class2` |
| 3 | Cloud shadows | `mask_class3` |
| 4 | Vegetation | `mask_class4` |
| 5 | Bare soils | `mask_class5` |
| 6 | Water | `mask_class6` |
| 7 | Cloud low probability | `mask_class7` |
| 8 | Cloud medium probability | `mask_class8` |
| 9 | Cloud high probability | `mask_class9` |
| 10 | Thin cirrus | `mask_class10` |
| 11 | Snow / ice | `mask_class11` |

For each image, `mask_cloud_and_shadows` builds a mask excluding every *checked* class (`scl.neq(class_value)`), applies it, then computes `percentage_valid_pixels = valid / total * 100` using `ee.Reducer.count()` over the AOI at `scale=10` (counting band `B1`). The collection is filtered with `ee.Filter.gte("percentage_valid_pixels", valid_pixel_threshold)`; the surviving `system:time_start` timestamps are then used (via `ee.Filter.inList`) to subset the **original, unmasked** collection - so SCL_filter only *selects* dates, it does not mask the returned imagery. Recorded as **SCL filter (Step 9)**.

Only applied when `local_pixel_limit > 0`.

In [8]:
# SCL class -> excluded? Mirrors the mask_classN.isChecked() booleans.
# Default below excludes clouds + shadows + no-data/defective + cirrus, which is
# the typical cloud-screening setup. Flip values to match your UI checkboxes.
scl_classes_behavior = {
    0: True,    # No data
    1: True,    # Saturated/defective
    2: False,   # Dark features
    3: True,    # Cloud shadows
    4: False,   # Vegetation
    5: False,   # Bare soils
    6: False,   # Water
    7: False,   # Cloud low probability
    8: True,    # Cloud medium probability
    9: True,    # Cloud high probability
    10: True,   # Thin cirrus
    11: False,  # Snow or ice
}


def SCL_filter(sentinel2, aoi, valid_pixel_threshold):
    """Filters by % valid pixels within the AOI (legacy SCL_filter, ~line 3808)."""
    print("Applying SCL filter...")

    def mask_cloud_and_shadows(image):
        scl = image.select("SCL")
        mask = ee.Image.constant(1)
        for class_value, include in scl_classes_behavior.items():
            if include:
                mask = mask.And(scl.neq(class_value))

        masked_image = image.updateMask(mask)

        total_pixels = (
            image.select(0)
            .reduceRegion(reducer=ee.Reducer.count(), geometry=aoi, scale=10)
            .get("B1")
        )
        valid_pixels = (
            masked_image.select(0)
            .reduceRegion(reducer=ee.Reducer.count(), geometry=aoi, scale=10)
            .get("B1")
        )
        percentage_valid = ee.Number(valid_pixels).divide(total_pixels).multiply(100)
        return masked_image.set("percentage_valid_pixels", percentage_valid)

    sentinel2_masked = sentinel2.map(mask_cloud_and_shadows)
    filtered_collection = sentinel2_masked.filter(
        ee.Filter.gte("percentage_valid_pixels", valid_pixel_threshold)
    )

    masked_timestamps = filtered_collection.aggregate_array("system:time_start").getInfo()
    sentinel2 = sentinel2.filter(
        ee.Filter.inList("system:time_start", ee.List(masked_timestamps))
    )

    size_after = sentinel2.size().getInfo()
    print("Collection size after SCL filter:", size_after)
    collection_info.append(f"Collection size after SCL filter (Step 9): {size_after}")
    collection_info_pt.append(
        f"Tamanho da cole\u00e7\u00e3o ap\u00f3s filtro SCL (Passo 9): {size_after}"
    )
    return sentinel2


if local_pixel_limit > 0:
    # Legacy passes the AOI geometry (after .first().geometry() upstream); the
    # reduceRegion here takes a geometry, so we use the buffered AOI geometry.
    aoi_geom_for_scl = aoi_fc.geometry()
    sentinel2 = SCL_filter(sentinel2, aoi_geom_for_scl, local_pixel_limit)
    assert sentinel2.size().getInfo() > 0, "SCL valid-pixel filter emptied the collection."
else:
    print("local_pixel_limit == 0 -> SCL valid-pixel filter skipped (matches legacy guard).")

Applying SCL filter...
Collection size after SCL filter: 142


## Step 4 - Apply the SCL class mask to imagery (`SCL_mask`)

Faithful to `SCL_mask` (~3886-3913). When the `mask` checkbox is on, this *hard-masks* the pixels of the selected SCL classes out of each image (so downstream index calculations ignore them). It collects the checked classes, builds `scl.neq(c0).And(scl.neq(c1))...`, and `updateMask`s each image. Unlike `SCL_filter` (which only selected dates), this changes pixel values of the returned collection.

It uses the same `mask_classN` checkbox booleans - reused here as `scl_classes_behavior`.

In [9]:
def SCL_mask(sentinel2, aoi):
    """Hard-masks selected SCL classes from imagery (legacy SCL_mask, ~line 3886)."""
    print("Applying SCL MASK...")
    selected_classes = [
        class_value
        for class_value, include in scl_classes_behavior.items()
        if include
    ]

    def mask_selected_classes(image):
        scl = image.select("SCL")
        mask = (
            scl.neq(selected_classes[0]) if selected_classes else ee.Image.constant(1)
        )
        for class_value in selected_classes[1:]:
            mask = mask.And(scl.neq(class_value))
        return image.updateMask(mask)

    return sentinel2.map(mask_selected_classes)


if mask_is_checked:
    sentinel2 = SCL_mask(sentinel2, aoi_fc.geometry())
    masked_count = sentinel2.size().getInfo()
    print(f"Collection size after SCL mask: {masked_count}")
    assert masked_count > 0, "SCL mask emptied the collection."
else:
    print("mask checkbox off -> SCL_mask skipped.")

Applying SCL MASK...
Collection size after SCL mask: 142


## Step 5 - Unique-day collection (`uniqueday_collection`)

Faithful to `uniqueday_collection` (~3704-3741). Sentinel-2 can produce multiple footprints for the same day; this keeps **one image per date**, preferring the highest `coverage_ratio` (set in Step 2). It aggregates the distinct `date` values, and for each date filters to that day and takes `.sort("coverage_ratio", False).first()`. Nulls are removed and the result is rebuilt as an `ImageCollection`. Recorded as the unique-dates stage.

> Note: `coverage_ratio` only exists if Step 2 ran (`coverage_threshold > 0`). If you skipped it, the sort is a no-op (still returns one image per date).

In [10]:
def uniqueday_collection(sentinel2):
    """One image per day, best AOI coverage (legacy uniqueday_collection, ~line 3704)."""
    print("Filtering to unique days using Earth Engine...")
    print("Original collection size:", sentinel2.size().getInfo())

    def process_date(date):
        date_start = ee.Date(date)
        date_end = date_start.advance(1, "day")
        daily_images = sentinel2.filterDate(date_start, date_end)
        best_image = daily_images.sort("coverage_ratio", False).first()
        return best_image

    dates = sentinel2.aggregate_array("date").distinct()
    unique_images = ee.List(dates.map(process_date))
    unique_images = unique_images.removeAll([None])
    filtered_collection = ee.ImageCollection(unique_images)

    size_unique = filtered_collection.size().getInfo()
    print(f"Collection size after keeping only unique dates: {size_unique}")
    collection_info.append(
        f"Collection size after keeping only unique dates: {size_unique}"
    )
    collection_info_pt.append(
        f"Tamanho da cole\u00e7\u00e3o ap\u00f3s manter apenas datas \u00fanicas: {size_unique}"
    )
    return filtered_collection


sentinel2 = uniqueday_collection(sentinel2)
final_count = sentinel2.size().getInfo()
print(f"\nFinal collection size: {final_count}")

Filtering to unique days using Earth Engine...
Original collection size: 142
Collection size after keeping only unique dates: 72

Final collection size: 72


## Step 6 - Inspect results: `collection_info` stages + per-image table

Print the accumulated `collection_info` metadata list (the same stage strings the plugin surfaces), then build a small pandas table of the surviving images with their `date` and (when present) `coverage_ratio`.

In [11]:
print("=== collection_info (EN) ===")
for line in collection_info:
    print("  -", line)

print("\n=== collection_info_pt (PT) ===")
for line in collection_info_pt:
    print("  -", line)

# Per-image summary table.
dates = sentinel2.aggregate_array("date").getInfo()
try:
    cover = sentinel2.aggregate_array("coverage_ratio").getInfo()
except Exception:
    cover = [None] * len(dates)

df = pd.DataFrame({"date": dates, "coverage_ratio": cover}).sort_values("date")
print(f"\nFinal unique-day images: {len(df)}")
df.reset_index(drop=True)

=== collection_info (EN) ===
  - Collection size before any filtering: 288
  - Collection size after tile cloud percentage filtering (Step 8): 172
  - Collection size after AOI overlap filter (Step 6): 170
  - Collection size before any filtering: 288
  - Collection size after tile cloud percentage filtering (Step 8): 172
  - Collection size after AOI overlap filter (Step 6): 162
  - Collection size after SCL filter (Step 9): 142
  - Collection size after keeping only unique dates: 72

=== collection_info_pt (PT) ===
  - Tamanho da coleção antes de qualquer filtro: 288
  - Tamanho da coleção após filtro de limite de nuvem na cena (Passo 8): 172
  - Tamanho da coleção após filtro de sobreposição com a AOI (Passo 6): 170
  - Tamanho da coleção antes de qualquer filtro: 288
  - Tamanho da coleção após filtro de limite de nuvem na cena (Passo 8): 172
  - Tamanho da coleção após filtro de sobreposição com a AOI (Passo 6): 162
  - Tamanho da coleção após filtro SCL (Passo 9): 142
  - Tamanho

,date,coverage_ratio
0,2023-02-08,1.000000
1,2023-02-20,0.978874
2,2023-02-25,1.000000
3,2023-03-02,0.995170
4,2023-03-20,1.000000
...,...,...
67,2023-11-12,1.000000
68,2023-12-10,1.000000
69,2023-12-20,1.000000
70,2023-12-25,1.000000


## (Optional) Interactive preview with geemap

If `geemap` is installed, preview the median true-color composite of the final masked collection over the AOI. Purely for visual sanity-checking - guarded so the notebook still runs top-to-bottom without geemap.

In [ ]:
if HAS_GEEMAP and final_count > 0:
    composite = sentinel2.median().clip(aoi)
    vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
    m = geemap.Map()
    m.centerObject(aoi, 12)
    m.addLayer(composite, vis, "S2 median (masked)")
    m.addLayer(aoi, {"color": "red"}, "AOI")
    display(m)
else:
    print("Skipping map preview (geemap unavailable or empty collection).")